In [0]:
from pyspark.sql import functions as F

cat = "databricks_catalog"
tables = {               # table: key column
    "orders":    "order_id",
    "customers": "customer_id",
    "products":  "product_id",
    "regions":   "region_id",
}

rows = []
for t, key in tables.items():
    bronze = spark.table(f"{cat}.bronze_schema.{t}")
    silver = spark.table(f"{cat}.silver_schema.{t}")
    rows.append((
        t,
        bronze.count(),
        silver.count(),
        silver.count() - silver.select(key).distinct().count(),   # duplicate keys
        silver.filter(F.col(key).isNull()).count(),               # missing keys
        "_rescued_data" in silver.columns,                        # should be False
    ))

display(spark.createDataFrame(rows, ["table", "bronze_rows", "silver_rows",
                                     "duplicate_keys", "null_keys", "has_rescued_data"]))

table,bronze_rows,silver_rows,duplicate_keys,null_keys,has_rescued_data
orders,10000,10000,0,0,false
customers,2000,2000,0,0,false
products,500,500,0,0,false
regions,4,4,0,0,false


In [0]:
orders    = spark.table(f"{cat}.silver_schema.orders")
customers = spark.table(f"{cat}.silver_schema.customers")
products  = spark.table(f"{cat}.silver_schema.products")

orphan_customers = orders.join(customers, "customer_id", "left_anti").count()
orphan_products  = orders.join(products,  "product_id",  "left_anti").count()

print("Orders with unknown customer:", orphan_customers)
print("Orders with unknown product: ", orphan_products)

Orders with unknown customer: 0
Orders with unknown product:  0


In [0]:
print("Orders with quantity <= 0:  ", orders.filter(F.col("quantity") <= 0).count())
print("Orders with amount < 0:     ", orders.filter(F.col("total_amount") < 0).count())
print("Products with price <= 0:   ", products.filter(F.col("price") <= 0).count())
print("Customers without '@' email:", customers.filter(~F.col("email").contains("@")).count())

display(orders.select(F.min("order_date").alias("first_order"),
                      F.max("order_date").alias("last_order")))

Orders with quantity <= 0:   0
Orders with amount < 0:      0
Products with price <= 0:    0
Customers without '@' email: 0


first_order,last_order
2023-01-01,2024-12-31


In [0]:
for t in tables:
    print(f"\n{t}")
    for f in spark.table(f"{cat}.silver_schema.{t}").schema.fields:
        print(f"  {f.name}: {f.dataType.simpleString()}")


orders
  order_id: string
  customer_id: string
  product_id: string
  order_date: date
  quantity: int
  total_amount: decimal(12,2)
  order_year: int
  order_month: int
  processed_at: timestamp

customers
  customer_id: string
  first_name: string
  last_name: string
  email: string
  city: string
  state: string
  email_domain: string
  full_name: string
  processed_at: timestamp

products
  product_id: string
  product_name: string
  category: string
  brand: string
  price: decimal(12,2)
  price_band: string
  discounted_price: decimal(12,2)
  processed_at: timestamp

regions
  region_id: string
  region: string
  processed_at: timestamp
